<div class="lesson-banner">
<span class="lesson-kicker">Python course · 2-hour lesson</span>
<p>Model stateful domain concepts with classes, dataclasses, composition, and explicit invariants.</p>
</div>

## Learning objectives

- Choose a class only when data and behavior belong together.
- Use instance, class, and static methods appropriately.
- Create concise value objects with dataclasses.
- Prefer composition over deep inheritance.

::: {.callout-note}
### How to use this notebook
Read the explanation, predict each result, run the code, change the inputs, and complete the practice before revealing the solution.
:::


## Objects protect invariants

A class is useful when an entity owns state and behavior across time. The constructor establishes a valid state; methods preserve it. A leading underscore marks implementation detail by convention. Properties can expose computed or validated attributes without leaking internal representation.


In [ ]:
class LearningPath:
    def __init__(self, name: str, total_lessons: int):
        if total_lessons <= 0:
            raise ValueError("total_lessons must be positive")
        self.name = name
        self.total_lessons = total_lessons
        self._completed = 0

    @property
    def progress(self) -> float:
        return self._completed / self.total_lessons

    def complete_lesson(self) -> None:
        if self._completed < self.total_lessons:
            self._completed += 1


path = LearningPath("Python", 18)
path.complete_lesson()
print(path.name, f"{path.progress:.1%}")


## Dataclasses for value-focused models

A dataclass generates initialization, representation, and equality methods from annotated fields. Use `frozen=True` for immutable value objects and `field(default_factory=...)` for mutable defaults. `__post_init__` can validate a completed object.


In [ ]:
from dataclasses import dataclass, field


@dataclass(frozen=True, slots=True)
class Course:
    code: str
    title: str
    hours: int
    tags: tuple[str, ...] = field(default_factory=tuple)

    def __post_init__(self):
        if self.hours <= 0:
            raise ValueError("hours must be positive")


python = Course("PY-101", "Python", 45, ("AI", "data"))
print(python)


## Composition and polymorphism

Composition builds an object from collaborators with narrow responsibilities. It is easier to replace a composed notifier or repository in tests than to untangle a deep inheritance hierarchy. Polymorphism means callers depend on behavior rather than a concrete class.


In [ ]:
class ConsoleNotifier:
    def send(self, message: str) -> None:
        print("NOTICE:", message)


class EnrollmentService:
    def __init__(self, notifier):
        self.notifier = notifier

    def enroll(self, learner: str, course: str) -> dict:
        record = {"learner": learner, "course": course}
        self.notifier.send(f"{learner} enrolled in {course}")
        return record


service = EnrollmentService(ConsoleNotifier())
print(service.enroll("Asha", "Python"))


## Worked example: immutable money value object

A value object centralizes currency and precision rules so calculations cannot silently mix incompatible values.


In [ ]:
from dataclasses import dataclass
from decimal import Decimal, ROUND_HALF_UP


@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: str = "INR"

    def __post_init__(self):
        rounded = self.amount.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        object.__setattr__(self, "amount", rounded)

    def __add__(self, other: "Money") -> "Money":
        if self.currency != other.currency:
            raise ValueError("currency mismatch")
        return Money(self.amount + other.amount, self.currency)


total = Money(Decimal("199.995")) + Money(Decimal("50"))
print(total)


## Practice lab

Complete these tasks without copying the solution. Test normal, boundary, and invalid inputs where relevant.

1. Create a `Learner` dataclass with a default empty skill set using `default_factory`.
2. Add a method that completes a lesson only once.
3. Compose a report service from a repository and formatter.
4. Explain why inheriting `ReportService` from `SqlRepository` would model the relationship poorly.

::: {.callout-important}
### Practice standard
Your answer should be readable, deterministic, and divided into small functions when the task contains more than one rule.
:::


## Suggested solution

Open the folded code only after attempting every task.


In [ ]:
from dataclasses import dataclass, field


@dataclass
class Learner:
    name: str
    skills: set[str] = field(default_factory=set)
    completed_lessons: set[str] = field(default_factory=set)

    def complete(self, lesson_id: str) -> bool:
        before = len(self.completed_lessons)
        self.completed_lessons.add(lesson_id)
        return len(self.completed_lessons) > before


class MemoryRepository:
    def load_scores(self):
        return [80, 90, 70]


class ReportService:
    def __init__(self, repository):
        self.repository = repository

    def average(self):
        scores = self.repository.load_scores()
        return sum(scores) / len(scores)


learner = Learner("Asha")
print(learner.complete("01"), learner.complete("01"))
print(ReportService(MemoryRepository()).average())


## Knowledge check

**1. When is a class better than functions?**

::: {.callout-note collapse="true"}
### Answer
When state and behavior form a cohesive concept with a lifecycle.
:::

**2. How do you define a safe mutable dataclass default?**

::: {.callout-note collapse="true"}
### Answer
Use `field(default_factory=...)`.
:::

**3. Why favor composition?**

::: {.callout-note collapse="true"}
### Answer
Dependencies remain replaceable and relationships stay explicit.
:::


## Recap

- Use classes for cohesive stateful models.
- Dataclasses reduce value-object boilerplate.
- Prefer shallow composition and explicit invariants.


<div class="lesson-nav">
<a href="08-modules-packages-environments.html"><i class="bi bi-arrow-left" aria-hidden="true"></i> Modules, Packages, and Environments</a>
<a href="10-pythonic-abstractions.html">Iterators, Generators, Decorators, and Context Managers <i class="bi bi-arrow-right" aria-hidden="true"></i></a>
</div>
